# Compile Season Data

Get tournament matchup matrix for specified season

In [1]:
season = 2024

playin_losers = (  # remove play-in losers from seeding data
    3357,  # Sacred Heart
    3162,  # Columbia
    3120,  # Auburn
    3221,  # Holy Cross
)

model_path = '../data/models/womens/2025_03_07_model.pkl'
data_path = '../data/models/womens/2025_03_07_data.parquet'

season

2024

### Previous Tournament Results

In [2]:
import pandas as pd

pd.set_option('display.max_columns', 100)

df = pd.read_parquet(r'..\data\preprocessed\womens_kaggle\tournament_results.parquet')

df = df.loc[df['Season'] == season, :].reset_index(drop=True)

df

,Season,TeamID,Team,Past Year Tournament Result,Past 4 Years Tournament Results
0,2024,3101,Abilene Chr,-1.0,-1.0
1,2024,3102,Air Force,-1.0,-1.0
2,2024,3103,Akron,-1.0,-1.0
3,2024,3104,Alabama,0.0,0.0
4,2024,3105,Alabama A&M,-1.0,-1.0
...,...,...,...,...,...
373,2024,3476,Stonehill,-1.0,-1.0
374,2024,3477,East Texas A&M,-1.0,-1.0
375,2024,3478,Le Moyne,-1.0,-1.0
376,2024,3479,Mercyhurst,-1.0,-1.0


### Barttorvik Ratings

In [3]:
df_barttorvik = pd.read_parquet(r'..\data\preprocessed\womens_barttorvik\barttorvik.parquet')

df_barttorvik = df_barttorvik.loc[df_barttorvik['Season'] == season, :].reset_index(drop=True)

df_barttorvik

,Season,TEAM,WIN%,ADJOE,ADJDE,ADJEM,BARTHAG,ADJ T.,WAB
0,2024,South Carolina,1.000000,122.5,73.2,49.3,0.9973,72.9,14.0
1,2024,Connecticut,0.848485,122.0,75.3,46.7,0.9961,71.5,9.3
2,2024,Texas,0.882353,120.9,76.2,44.7,0.9951,70.6,9.5
3,2024,UCLA,0.806452,117.1,75.7,41.4,0.9934,70.5,9.6
4,2024,Stanford,0.848485,121.6,79.1,42.5,0.9929,69.1,10.1
...,...,...,...,...,...,...,...,...,...
355,2024,Chicago St.,0.000000,76.3,111.4,-35.1,0.0126,75.9,-23.3
356,2024,Stonehill,0.133333,72.9,108.2,-35.3,0.0105,69.5,-24.7
357,2024,Wagner,0.222222,71.3,107.8,-36.5,0.0086,70.6,-20.1
358,2024,South Carolina St.,0.066667,70.5,107.1,-36.6,0.0081,68.1,-25.8


In [4]:
df_spellings = pd.read_csv(
    r'..\data\unprocessed\kaggle\WTeamSpellings.csv', 
    encoding='cp1252'  # fixes issue with fancy quotes
)

df_spellings.loc[df_spellings.shape[0]] = ['fdu', 3192]

df_spellings

,TeamNameSpelling,TeamID
0,a&m-corpus chris,3394
1,a&m-corpus christi,3394
2,abilene chr,3101
3,abilene christian,3101
4,abilene-christian,3101
...,...,...
1171,youngstown st.,3464
1172,youngstown state,3464
1173,youngstown-st,3464
1174,youngstown-state,3464


In [5]:
spelling_to_id = dict(zip(df_spellings['TeamNameSpelling'], df_spellings['TeamID']))

len(spelling_to_id)

1170

In [6]:
from fuzzywuzzy.fuzz import token_sort_ratio
from fuzzywuzzy import process
from tqdm.autonotebook import tqdm

def match_names(team_spellings, new_data_teams):
    df_match = pd.DataFrame(
        [
            [
                new_data_team,
                *process.extract(
                    new_data_team,
                    team_spellings,
                    scorer=token_sort_ratio,
                    limit=1
                )[0][:2]
            ] for new_data_team in tqdm(new_data_teams)
        ],
        columns=['New Data Team', 'Team Spelling', 'Match Score']
    ).sort_values('Match Score', ignore_index=True)

    team_to_spelling = dict(zip(df_match['New Data Team'], df_match['Team Spelling']))

    return df_match, team_to_spelling

C:\Users\mhugh\AppData\Local\Temp\ipykernel_10172\2578028529.py:3: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


In [7]:
df_match, team_to_spelling = match_names(df_spellings['TeamNameSpelling'].unique(), df_barttorvik['TEAM'].unique())

df_match.head(25)

  0%|          | 0/360 [00:00<?, ?it/s]

,New Data Team,Team Spelling,Match Score
0,Queens,queens (nc),80
1,UT Rio Grande Valley,texas rio grande valley,88
2,Saint Francis,saint francis (ny),90
3,Cal St. Bakersfield,cal state bakersfield,92
4,Southeast Missouri St.,southeast missouri state,93
5,Mississippi Valley St.,mississippi valley state,93
6,Texas A&M Corpus Chris,texas a&m-corpus christi,96
7,Cornell,cornell,100
8,Southern,southern,100
9,Dayton,dayton,100


In [8]:
df_barttorvik.insert(1, 'TeamID', df_barttorvik['TEAM'].map(team_to_spelling).map(spelling_to_id))

df_barttorvik

,Season,TeamID,TEAM,WIN%,ADJOE,ADJDE,ADJEM,BARTHAG,ADJ T.,WAB
0,2024,3376,South Carolina,1.000000,122.5,73.2,49.3,0.9973,72.9,14.0
1,2024,3163,Connecticut,0.848485,122.0,75.3,46.7,0.9961,71.5,9.3
2,2024,3400,Texas,0.882353,120.9,76.2,44.7,0.9951,70.6,9.5
3,2024,3417,UCLA,0.806452,117.1,75.7,41.4,0.9934,70.5,9.6
4,2024,3390,Stanford,0.848485,121.6,79.1,42.5,0.9929,69.1,10.1
...,...,...,...,...,...,...,...,...,...,...
355,2024,3152,Chicago St.,0.000000,76.3,111.4,-35.1,0.0126,75.9,-23.3
356,2024,3476,Stonehill,0.133333,72.9,108.2,-35.3,0.0105,69.5,-24.7
357,2024,3447,Wagner,0.222222,71.3,107.8,-36.5,0.0086,70.6,-20.1
358,2024,3354,South Carolina St.,0.066667,70.5,107.1,-36.6,0.0081,68.1,-25.8


In [9]:
df = pd.merge(
    df,
    df_barttorvik.drop(columns=['TEAM']),
    how='left',
    on=['Season', 'TeamID']
)

df

,Season,TeamID,Team,Past Year Tournament Result,Past 4 Years Tournament Results,WIN%,ADJOE,ADJDE,ADJEM,BARTHAG,ADJ T.,WAB
0,2024,3101,Abilene Chr,-1.0,-1.0,0.407407,95.1,98.3,-3.2,0.4052,68.4,-14.0
1,2024,3102,Air Force,-1.0,-1.0,0.433333,90.1,93.7,-3.6,0.3892,70.6,-12.5
2,2024,3103,Akron,-1.0,-1.0,0.379310,87.4,98.7,-11.3,0.1973,67.2,-16.1
3,2024,3104,Alabama,0.0,0.0,0.718750,107.9,84.0,23.9,0.9468,69.7,2.0
4,2024,3105,Alabama A&M,-1.0,-1.0,0.466667,83.6,96.9,-13.3,0.1559,68.4,-14.4
...,...,...,...,...,...,...,...,...,...,...,...,...
373,2024,3476,Stonehill,-1.0,-1.0,0.133333,72.9,108.2,-35.3,0.0105,69.5,-24.7
374,2024,3477,East Texas A&M,-1.0,-1.0,0.448276,88.8,101.9,-13.1,0.1704,76.3,-13.3
375,2024,3478,Le Moyne,-1.0,-1.0,0.562500,82.7,97.2,-14.5,0.1339,66.5,-9.8
376,2024,3479,Mercyhurst,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Past Seasons

In [10]:
df_ps = pd.read_parquet(r'..\data\preprocessed\womens_past_seasons\past_seasons_ratings.parquet')

df_ps = df_ps.loc[df_ps['Season'] == season, :].reset_index(drop=True)

df_ps

,Season,Team,Past Year Efficiency Margin,Past 4 Years Efficiency Margin
0,2024,Abilene Christian,-0.002761,-0.036218
1,2024,Air Force,-0.019225,-0.021754
2,2024,Akron,-0.040582,-0.020406
3,2024,Alabama,0.274209,0.217993
4,2024,Alabama A&M,-0.168682,-0.098402
...,...,...,...,...
358,2024,Wright State,-0.173424,-0.053027
359,2024,Wyoming,0.098008,0.069632
360,2024,Xavier,-0.059519,-0.060891
361,2024,Yale,-0.035396,0.026920


In [11]:
df_match, team_to_spelling = match_names(df_spellings['TeamNameSpelling'].unique(), df_ps['Team'].unique())

df_match.head(25)

  0%|          | 0/363 [00:00<?, ?it/s]

,New Data Team,Team Spelling,Match Score
0,Hartford Hawks,hartford,73
1,St. Francis (NY) Terriers,st francis (ny),74
2,Savannah State Tigers,savannah state,80
3,Abilene Christian,abilene christian,100
4,Quinnipiac,quinnipiac,100
5,Queens (NC),queens (nc),100
6,Purdue Fort Wayne,purdue fort wayne,100
7,Purdue,purdue,100
8,Providence,providence,100
9,Princeton,princeton,100


In [12]:
df_ps.insert(1, 'TeamID', df_ps['Team'].map(team_to_spelling).map(spelling_to_id))

df_ps

,Season,TeamID,Team,Past Year Efficiency Margin,Past 4 Years Efficiency Margin
0,2024,3101,Abilene Christian,-0.002761,-0.036218
1,2024,3102,Air Force,-0.019225,-0.021754
2,2024,3103,Akron,-0.040582,-0.020406
3,2024,3104,Alabama,0.274209,0.217993
4,2024,3105,Alabama A&M,-0.168682,-0.098402
...,...,...,...,...,...
358,2024,3460,Wright State,-0.173424,-0.053027
359,2024,3461,Wyoming,0.098008,0.069632
360,2024,3462,Xavier,-0.059519,-0.060891
361,2024,3463,Yale,-0.035396,0.026920


In [13]:
df = pd.merge(
    df,
    df_ps.drop(columns=['Team']),
    how='left',
    on=['Season', 'TeamID']
)

df

,Season,TeamID,Team,Past Year Tournament Result,Past 4 Years Tournament Results,WIN%,ADJOE,ADJDE,ADJEM,BARTHAG,ADJ T.,WAB,Past Year Efficiency Margin,Past 4 Years Efficiency Margin
0,2024,3101,Abilene Chr,-1.0,-1.0,0.407407,95.1,98.3,-3.2,0.4052,68.4,-14.0,-0.002761,-0.036218
1,2024,3102,Air Force,-1.0,-1.0,0.433333,90.1,93.7,-3.6,0.3892,70.6,-12.5,-0.019225,-0.021754
2,2024,3103,Akron,-1.0,-1.0,0.379310,87.4,98.7,-11.3,0.1973,67.2,-16.1,-0.040582,-0.020406
3,2024,3104,Alabama,0.0,0.0,0.718750,107.9,84.0,23.9,0.9468,69.7,2.0,0.274209,0.217993
4,2024,3105,Alabama A&M,-1.0,-1.0,0.466667,83.6,96.9,-13.3,0.1559,68.4,-14.4,-0.168682,-0.098402
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
373,2024,3476,Stonehill,-1.0,-1.0,0.133333,72.9,108.2,-35.3,0.0105,69.5,-24.7,-0.234824,NaN
374,2024,3477,East Texas A&M,-1.0,-1.0,0.448276,88.8,101.9,-13.1,0.1704,76.3,-13.3,-0.116192,NaN
375,2024,3478,Le Moyne,-1.0,-1.0,0.562500,82.7,97.2,-14.5,0.1339,66.5,-9.8,NaN,NaN
376,2024,3479,Mercyhurst,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [14]:
df.loc[df['Past Year Efficiency Margin'].isna(), :]

,Season,TeamID,Team,Past Year Tournament Result,Past 4 Years Tournament Results,WIN%,ADJOE,ADJDE,ADJEM,BARTHAG,ADJ T.,WAB,Past Year Efficiency Margin,Past 4 Years Efficiency Margin
8,2024,3109,Alliant Intl,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
17,2024,3118,Armstrong St,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20,2024,3121,Augusta,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
27,2024,3128,Birmingham So,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
33,2024,3134,Brooklyn,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
46,2024,3147,Centenary,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
113,2024,3215,Hardin-Simmons,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
187,2024,3289,Morris Brown,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
200,2024,3302,NE Illinois,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
225,2024,3327,Okla City,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [15]:
df.loc[df['Past Year Efficiency Margin'].isna() & (df['Past 4 Years Tournament Results'] > -1.0), :]

,Season,TeamID,Team,Past Year Tournament Result,Past 4 Years Tournament Results,WIN%,ADJOE,ADJDE,ADJEM,BARTHAG,ADJ T.,WAB,Past Year Efficiency Margin,Past 4 Years Efficiency Margin


### My Rankings

In [16]:
df_rankings = pd.read_parquet(fr'..\data\preprocessed\womens_my_rankings\my_rankings_{season}.parquet')

df_rankings.insert(0, 'Season', season)

df_rankings.drop(columns=['Strength'], inplace=True)

df_rankings

,Season,Team,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo
0,2024,South Carolina,5.348664,0.540655,1.199302,0.658648,74.409140
1,2024,Connecticut,3.913262,0.504011,1.189835,0.685824,72.866365
2,2024,Texas,3.870221,0.456575,1.177144,0.720569,71.739541
3,2024,UCLA,3.817370,0.426926,1.138986,0.712061,71.853088
4,2024,Southern California,3.746376,0.373978,1.127198,0.753220,70.859265
...,...,...,...,...,...,...,...
355,2024,South Carolina State,-3.170798,-0.377634,0.685217,1.062851,69.006102
356,2024,Wagner,-3.192290,-0.354522,0.689296,1.043818,71.754876
357,2024,Long Island University,-3.272119,-0.320304,0.746029,1.066333,71.291510
358,2024,Stonehill,-3.305148,-0.371073,0.696827,1.067900,70.574844


In [17]:
df_match, team_to_spelling = match_names(df_spellings['TeamNameSpelling'].unique(), df_rankings['Team'].unique())

df_match.head(25)

  0%|          | 0/360 [00:00<?, ?it/s]

,New Data Team,Team Spelling,Match Score
0,South Carolina,south carolina,100
1,Le Moyne,le moyne,100
2,Elon,elon,100
3,Loyola (MD),loyola (md),100
4,UC San Diego,uc san diego,100
5,Yale,yale,100
6,Idaho,idaho,100
7,Howard,howard,100
8,Iona,iona,100
9,Youngstown State,youngstown state,100


In [18]:
df_rankings.insert(1, 'TeamID', df_rankings['Team'].map(team_to_spelling).map(spelling_to_id))

df_rankings

,Season,TeamID,Team,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo
0,2024,3376,South Carolina,5.348664,0.540655,1.199302,0.658648,74.409140
1,2024,3163,Connecticut,3.913262,0.504011,1.189835,0.685824,72.866365
2,2024,3400,Texas,3.870221,0.456575,1.177144,0.720569,71.739541
3,2024,3417,UCLA,3.817370,0.426926,1.138986,0.712061,71.853088
4,2024,3425,Southern California,3.746376,0.373978,1.127198,0.753220,70.859265
...,...,...,...,...,...,...,...,...
355,2024,3354,South Carolina State,-3.170798,-0.377634,0.685217,1.062851,69.006102
356,2024,3447,Wagner,-3.192290,-0.354522,0.689296,1.043818,71.754876
357,2024,3254,Long Island University,-3.272119,-0.320304,0.746029,1.066333,71.291510
358,2024,3476,Stonehill,-3.305148,-0.371073,0.696827,1.067900,70.574844


In [19]:
df_rankings.loc[df_rankings['TeamID'].isna(), :]

,Season,TeamID,Team,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo


In [20]:
df = pd.merge(
    df,
    df_rankings.drop(columns=['Team']),
    how='left',
    on=['Season', 'TeamID']
)

df

,Season,TeamID,Team,Past Year Tournament Result,Past 4 Years Tournament Results,WIN%,ADJOE,ADJDE,ADJEM,BARTHAG,ADJ T.,WAB,Past Year Efficiency Margin,Past 4 Years Efficiency Margin,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo
0,2024,3101,Abilene Chr,-1.0,-1.0,0.407407,95.1,98.3,-3.2,0.4052,68.4,-14.0,-0.002761,-0.036218,-0.836132,-0.030567,0.935291,0.965858,69.429008
1,2024,3102,Air Force,-1.0,-1.0,0.433333,90.1,93.7,-3.6,0.3892,70.6,-12.5,-0.019225,-0.021754,-0.546818,-0.046346,0.884672,0.931018,71.498056
2,2024,3103,Akron,-1.0,-1.0,0.379310,87.4,98.7,-11.3,0.1973,67.2,-16.1,-0.040582,-0.020406,-1.082640,-0.112023,0.856389,0.968412,68.420028
3,2024,3104,Alabama,0.0,0.0,0.718750,107.9,84.0,23.9,0.9468,69.7,2.0,0.274209,0.217993,2.390814,0.260254,1.063672,0.803417,71.089116
4,2024,3105,Alabama A&M,-1.0,-1.0,0.466667,83.6,96.9,-13.3,0.1559,68.4,-14.4,-0.168682,-0.098402,-1.433353,-0.133374,0.818854,0.952228,69.630643
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
373,2024,3476,Stonehill,-1.0,-1.0,0.133333,72.9,108.2,-35.3,0.0105,69.5,-24.7,-0.234824,NaN,-3.305148,-0.371073,0.696827,1.067900,70.574844
374,2024,3477,East Texas A&M,-1.0,-1.0,0.448276,88.8,101.9,-13.1,0.1704,76.3,-13.3,-0.116192,NaN,-0.767316,-0.136922,0.862589,0.999511,77.283300
375,2024,3478,Le Moyne,-1.0,-1.0,0.562500,82.7,97.2,-14.5,0.1339,66.5,-9.8,NaN,NaN,-0.988127,-0.134376,0.817137,0.951513,67.530652
376,2024,3479,Mercyhurst,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [21]:
df.loc[df['Rating'].isna(), :]

,Season,TeamID,Team,Past Year Tournament Result,Past 4 Years Tournament Results,WIN%,ADJOE,ADJDE,ADJEM,BARTHAG,ADJ T.,WAB,Past Year Efficiency Margin,Past 4 Years Efficiency Margin,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo
8,2024,3109,Alliant Intl,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
17,2024,3118,Armstrong St,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20,2024,3121,Augusta,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
27,2024,3128,Birmingham So,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
33,2024,3134,Brooklyn,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
46,2024,3147,Centenary,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
113,2024,3215,Hardin-Simmons,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
114,2024,3216,Hartford,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.505933,-0.326890,NaN,NaN,NaN,NaN,NaN
187,2024,3289,Morris Brown,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
200,2024,3302,NE Illinois,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Starters

In [22]:
df_starters = pd.read_parquet(fr'..\data\preprocessed\womens_starters\starters_{season}.parquet')

df_starters.insert(0, 'Season', season)

df_starters.rename(columns={'Rating': 'Starters'}, inplace=True)

df_starters

,Season,Team,Starters
0,2024,South Carolina,0.965694
1,2024,Texas,0.739441
2,2024,Southern California,0.733047
3,2024,UCLA,0.718998
4,2024,Gonzaga,0.712503
...,...,...,...
355,2024,Houston Christian,-0.531501
356,2024,Western Carolina,-0.533933
357,2024,South Carolina State,-0.555766
358,2024,McNeese State,-0.588590


In [23]:
df_match, team_to_spelling = match_names(df_spellings['TeamNameSpelling'].unique(), df_starters['Team'].unique())

df_match.head(25)

  0%|          | 0/360 [00:00<?, ?it/s]

,New Data Team,Team Spelling,Match Score
0,South Carolina,south carolina,100
1,George Washington,george washington,100
2,Air Force,air force,100
3,Alabama A&M,alabama a&m,100
4,Northern Colorado,northern colorado,100
5,Lehigh,lehigh,100
6,Army,army,100
7,Loyola (MD),loyola (md),100
8,Western Kentucky,western kentucky,100
9,North Carolina Central,north carolina central,100


In [24]:
df_starters.insert(1, 'TeamID', df_starters['Team'].map(team_to_spelling).map(spelling_to_id))

df_starters

,Season,TeamID,Team,Starters
0,2024,3376,South Carolina,0.965694
1,2024,3400,Texas,0.739441
2,2024,3425,Southern California,0.733047
3,2024,3417,UCLA,0.718998
4,2024,3211,Gonzaga,0.712503
...,...,...,...,...
355,2024,3223,Houston Christian,-0.531501
356,2024,3441,Western Carolina,-0.533933
357,2024,3354,South Carolina State,-0.555766
358,2024,3270,McNeese State,-0.588590


In [25]:
df_starters.loc[df_starters['TeamID'].isna(), :]

,Season,TeamID,Team,Starters


In [26]:
df = pd.merge(
    df,
    df_starters.drop(columns=['Team']),
    how='left',
    on=['Season', 'TeamID']
)

df

,Season,TeamID,Team,Past Year Tournament Result,Past 4 Years Tournament Results,WIN%,ADJOE,ADJDE,ADJEM,BARTHAG,ADJ T.,WAB,Past Year Efficiency Margin,Past 4 Years Efficiency Margin,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo,Starters
0,2024,3101,Abilene Chr,-1.0,-1.0,0.407407,95.1,98.3,-3.2,0.4052,68.4,-14.0,-0.002761,-0.036218,-0.836132,-0.030567,0.935291,0.965858,69.429008,-0.071268
1,2024,3102,Air Force,-1.0,-1.0,0.433333,90.1,93.7,-3.6,0.3892,70.6,-12.5,-0.019225,-0.021754,-0.546818,-0.046346,0.884672,0.931018,71.498056,-0.141134
2,2024,3103,Akron,-1.0,-1.0,0.379310,87.4,98.7,-11.3,0.1973,67.2,-16.1,-0.040582,-0.020406,-1.082640,-0.112023,0.856389,0.968412,68.420028,-0.164349
3,2024,3104,Alabama,0.0,0.0,0.718750,107.9,84.0,23.9,0.9468,69.7,2.0,0.274209,0.217993,2.390814,0.260254,1.063672,0.803417,71.089116,0.389101
4,2024,3105,Alabama A&M,-1.0,-1.0,0.466667,83.6,96.9,-13.3,0.1559,68.4,-14.4,-0.168682,-0.098402,-1.433353,-0.133374,0.818854,0.952228,69.630643,-0.140222
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
373,2024,3476,Stonehill,-1.0,-1.0,0.133333,72.9,108.2,-35.3,0.0105,69.5,-24.7,-0.234824,NaN,-3.305148,-0.371073,0.696827,1.067900,70.574844,-0.463501
374,2024,3477,East Texas A&M,-1.0,-1.0,0.448276,88.8,101.9,-13.1,0.1704,76.3,-13.3,-0.116192,NaN,-0.767316,-0.136922,0.862589,0.999511,77.283300,-0.009010
375,2024,3478,Le Moyne,-1.0,-1.0,0.562500,82.7,97.2,-14.5,0.1339,66.5,-9.8,NaN,NaN,-0.988127,-0.134376,0.817137,0.951513,67.530652,0.055646
376,2024,3479,Mercyhurst,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [27]:
df.loc[df['Starters'].isna(), :]

,Season,TeamID,Team,Past Year Tournament Result,Past 4 Years Tournament Results,WIN%,ADJOE,ADJDE,ADJEM,BARTHAG,ADJ T.,WAB,Past Year Efficiency Margin,Past 4 Years Efficiency Margin,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo,Starters
8,2024,3109,Alliant Intl,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
17,2024,3118,Armstrong St,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20,2024,3121,Augusta,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
27,2024,3128,Birmingham So,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
33,2024,3134,Brooklyn,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
46,2024,3147,Centenary,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
113,2024,3215,Hardin-Simmons,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
114,2024,3216,Hartford,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.505933,-0.326890,NaN,NaN,NaN,NaN,NaN,NaN
187,2024,3289,Morris Brown,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
200,2024,3302,NE Illinois,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Openskill Ratings

In [28]:
df_os = pd.read_parquet(fr'..\data\preprocessed\womens_os_rankings\os_rankings_{season}.parquet')

df_os.insert(0, 'Season', season)

df_os.drop(columns=['Sigma'], inplace=True)

df_os

,Season,Team,Mu,OS Rating
0,2024,South Carolina,59.034668,45.727392
1,2024,Texas,53.074235,40.494205
2,2024,Connecticut,51.460005,39.055079
3,2024,Iowa,51.402284,38.731977
4,2024,Southern California,51.110974,38.512240
...,...,...,...,...
355,2024,Alabama State,1.718860,-12.127996
356,2024,McNeese State,2.277615,-12.622216
357,2024,Houston Christian,1.520705,-12.900479
358,2024,South Carolina State,-0.539184,-14.171589


In [29]:
df_match, team_to_spelling = match_names(df_spellings['TeamNameSpelling'].unique(), df_os['Team'].unique())

df_match.head(25)

  0%|          | 0/360 [00:00<?, ?it/s]

,New Data Team,Team Spelling,Match Score
0,South Carolina,south carolina,100
1,Delaware,delaware,100
2,Quinnipiac,quinnipiac,100
3,Wichita State,wichita state,100
4,Loyola Marymount,loyola marymount,100
5,Abilene Christian,abilene christian,100
6,St. Thomas,st. thomas,100
7,New Mexico State,new mexico state,100
8,Eastern Illinois,eastern illinois,100
9,Radford,radford,100


In [30]:
df_os.insert(1, 'TeamID', df_os['Team'].map(team_to_spelling).map(spelling_to_id))

df_os

,Season,TeamID,Team,Mu,OS Rating
0,2024,3376,South Carolina,59.034668,45.727392
1,2024,3400,Texas,53.074235,40.494205
2,2024,3163,Connecticut,51.460005,39.055079
3,2024,3234,Iowa,51.402284,38.731977
4,2024,3425,Southern California,51.110974,38.512240
...,...,...,...,...,...
355,2024,3106,Alabama State,1.718860,-12.127996
356,2024,3270,McNeese State,2.277615,-12.622216
357,2024,3223,Houston Christian,1.520705,-12.900479
358,2024,3354,South Carolina State,-0.539184,-14.171589


In [31]:
df = pd.merge(
    df,
    df_os.drop(columns=['Team']),
    how='left',
    on=['Season', 'TeamID']
)

df

,Season,TeamID,Team,Past Year Tournament Result,Past 4 Years Tournament Results,WIN%,ADJOE,ADJDE,ADJEM,BARTHAG,ADJ T.,WAB,Past Year Efficiency Margin,Past 4 Years Efficiency Margin,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo,Starters,Mu,OS Rating
0,2024,3101,Abilene Chr,-1.0,-1.0,0.407407,95.1,98.3,-3.2,0.4052,68.4,-14.0,-0.002761,-0.036218,-0.836132,-0.030567,0.935291,0.965858,69.429008,-0.071268,18.501742,5.651852
1,2024,3102,Air Force,-1.0,-1.0,0.433333,90.1,93.7,-3.6,0.3892,70.6,-12.5,-0.019225,-0.021754,-0.546818,-0.046346,0.884672,0.931018,71.498056,-0.141134,20.312118,7.604695
2,2024,3103,Akron,-1.0,-1.0,0.379310,87.4,98.7,-11.3,0.1973,67.2,-16.1,-0.040582,-0.020406,-1.082640,-0.112023,0.856389,0.968412,68.420028,-0.164349,17.960257,4.853112
3,2024,3104,Alabama,0.0,0.0,0.718750,107.9,84.0,23.9,0.9468,69.7,2.0,0.274209,0.217993,2.390814,0.260254,1.063672,0.803417,71.089116,0.389101,41.428647,29.490924
4,2024,3105,Alabama A&M,-1.0,-1.0,0.466667,83.6,96.9,-13.3,0.1559,68.4,-14.4,-0.168682,-0.098402,-1.433353,-0.133374,0.818854,0.952228,69.630643,-0.140222,17.055286,4.652751
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
373,2024,3476,Stonehill,-1.0,-1.0,0.133333,72.9,108.2,-35.3,0.0105,69.5,-24.7,-0.234824,NaN,-3.305148,-0.371073,0.696827,1.067900,70.574844,-0.463501,2.006163,-10.928439
374,2024,3477,East Texas A&M,-1.0,-1.0,0.448276,88.8,101.9,-13.1,0.1704,76.3,-13.3,-0.116192,NaN,-0.767316,-0.136922,0.862589,0.999511,77.283300,-0.009010,20.249666,7.487434
375,2024,3478,Le Moyne,-1.0,-1.0,0.562500,82.7,97.2,-14.5,0.1339,66.5,-9.8,NaN,NaN,-0.988127,-0.134376,0.817137,0.951513,67.530652,0.055646,23.057138,10.392918
376,2024,3479,Mercyhurst,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [32]:
df.loc[df['OS Rating'].isna(), :]

,Season,TeamID,Team,Past Year Tournament Result,Past 4 Years Tournament Results,WIN%,ADJOE,ADJDE,ADJEM,BARTHAG,ADJ T.,WAB,Past Year Efficiency Margin,Past 4 Years Efficiency Margin,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo,Starters,Mu,OS Rating
8,2024,3109,Alliant Intl,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
17,2024,3118,Armstrong St,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20,2024,3121,Augusta,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
27,2024,3128,Birmingham So,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
33,2024,3134,Brooklyn,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
46,2024,3147,Centenary,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
113,2024,3215,Hardin-Simmons,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
114,2024,3216,Hartford,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.505933,-0.326890,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
187,2024,3289,Morris Brown,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
200,2024,3302,NE Illinois,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Standard Stats

In [33]:
df_ss = pd.read_parquet('../data/preprocessed/womens_standard_stats/standard_stats.parquet')

df_ss = df_ss.loc[df_ss['Season'] == season, :].reset_index(drop=True)

df_ss

,Season,Team,Team Win%,Team EFG%,Opponent EFG%,Team TOR,Team ORBR,Team FTR,Opponent FTR
0,2024,Abilene Christian,0.407407,0.478077,0.490781,21.232178,14.880053,20.625826,20.122552
1,2024,Air Force,0.433333,0.438547,0.484612,19.250295,12.406726,16.749504,29.145055
2,2024,Akron,0.379310,0.437227,0.477698,21.734627,13.822299,19.185600,19.213234
3,2024,Alabama,0.718750,0.496489,0.428012,19.892904,15.069715,22.813074,18.450030
4,2024,Alabama A&M,0.466667,0.409847,0.441112,23.103184,16.513021,21.612872,24.870078
...,...,...,...,...,...,...,...,...,...
355,2024,Wright State,0.516129,0.477860,0.488562,18.043420,9.488515,21.044587,21.790551
356,2024,Wyoming,0.517241,0.507442,0.464675,20.152826,9.383333,17.073582,17.922837
357,2024,Xavier,0.035714,0.420328,0.513675,25.643612,8.332792,13.540131,21.670300
358,2024,Yale,0.296296,0.422138,0.493458,19.933187,15.534023,17.563822,25.764921


In [34]:
df_match, team_to_spelling = match_names(df_spellings['TeamNameSpelling'].unique(), df_ss['Team'].unique())

df_match.head(25)

  0%|          | 0/360 [00:00<?, ?it/s]

,New Data Team,Team Spelling,Match Score
0,Abilene Christian,abilene christian,100
1,Queens (NC),queens (nc),100
2,Purdue Fort Wayne,purdue fort wayne,100
3,Purdue,purdue,100
4,Providence,providence,100
5,Princeton,princeton,100
6,Presbyterian,presbyterian,100
7,Prairie View,prairie view,100
8,Portland State,portland state,100
9,Quinnipiac,quinnipiac,100


In [35]:
df_ss.insert(1, 'TeamID', df_ss['Team'].map(team_to_spelling).map(spelling_to_id))

df_ss

,Season,TeamID,Team,Team Win%,Team EFG%,Opponent EFG%,Team TOR,Team ORBR,Team FTR,Opponent FTR
0,2024,3101,Abilene Christian,0.407407,0.478077,0.490781,21.232178,14.880053,20.625826,20.122552
1,2024,3102,Air Force,0.433333,0.438547,0.484612,19.250295,12.406726,16.749504,29.145055
2,2024,3103,Akron,0.379310,0.437227,0.477698,21.734627,13.822299,19.185600,19.213234
3,2024,3104,Alabama,0.718750,0.496489,0.428012,19.892904,15.069715,22.813074,18.450030
4,2024,3105,Alabama A&M,0.466667,0.409847,0.441112,23.103184,16.513021,21.612872,24.870078
...,...,...,...,...,...,...,...,...,...,...
355,2024,3460,Wright State,0.516129,0.477860,0.488562,18.043420,9.488515,21.044587,21.790551
356,2024,3461,Wyoming,0.517241,0.507442,0.464675,20.152826,9.383333,17.073582,17.922837
357,2024,3462,Xavier,0.035714,0.420328,0.513675,25.643612,8.332792,13.540131,21.670300
358,2024,3463,Yale,0.296296,0.422138,0.493458,19.933187,15.534023,17.563822,25.764921


In [36]:
df = pd.merge(
    df,
    df_ss.drop(columns=['Team']),
    how='left',
    on=['Season', 'TeamID']
)

df

,Season,TeamID,Team,Past Year Tournament Result,Past 4 Years Tournament Results,WIN%,ADJOE,ADJDE,ADJEM,BARTHAG,ADJ T.,WAB,Past Year Efficiency Margin,Past 4 Years Efficiency Margin,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo,Starters,Mu,OS Rating,Team Win%,Team EFG%,Opponent EFG%,Team TOR,Team ORBR,Team FTR,Opponent FTR
0,2024,3101,Abilene Chr,-1.0,-1.0,0.407407,95.1,98.3,-3.2,0.4052,68.4,-14.0,-0.002761,-0.036218,-0.836132,-0.030567,0.935291,0.965858,69.429008,-0.071268,18.501742,5.651852,0.407407,0.478077,0.490781,21.232178,14.880053,20.625826,20.122552
1,2024,3102,Air Force,-1.0,-1.0,0.433333,90.1,93.7,-3.6,0.3892,70.6,-12.5,-0.019225,-0.021754,-0.546818,-0.046346,0.884672,0.931018,71.498056,-0.141134,20.312118,7.604695,0.433333,0.438547,0.484612,19.250295,12.406726,16.749504,29.145055
2,2024,3103,Akron,-1.0,-1.0,0.379310,87.4,98.7,-11.3,0.1973,67.2,-16.1,-0.040582,-0.020406,-1.082640,-0.112023,0.856389,0.968412,68.420028,-0.164349,17.960257,4.853112,0.379310,0.437227,0.477698,21.734627,13.822299,19.185600,19.213234
3,2024,3104,Alabama,0.0,0.0,0.718750,107.9,84.0,23.9,0.9468,69.7,2.0,0.274209,0.217993,2.390814,0.260254,1.063672,0.803417,71.089116,0.389101,41.428647,29.490924,0.718750,0.496489,0.428012,19.892904,15.069715,22.813074,18.450030
4,2024,3105,Alabama A&M,-1.0,-1.0,0.466667,83.6,96.9,-13.3,0.1559,68.4,-14.4,-0.168682,-0.098402,-1.433353,-0.133374,0.818854,0.952228,69.630643,-0.140222,17.055286,4.652751,0.466667,0.409847,0.441112,23.103184,16.513021,21.612872,24.870078
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
373,2024,3476,Stonehill,-1.0,-1.0,0.133333,72.9,108.2,-35.3,0.0105,69.5,-24.7,-0.234824,NaN,-3.305148,-0.371073,0.696827,1.067900,70.574844,-0.463501,2.006163,-10.928439,0.133333,0.391396,0.478187,25.925883,12.777546,16.302205,15.915717
374,2024,3477,East Texas A&M,-1.0,-1.0,0.448276,88.8,101.9,-13.1,0.1704,76.3,-13.3,-0.116192,NaN,-0.767316,-0.136922,0.862589,0.999511,77.283300,-0.009010,20.249666,7.487434,0.448276,0.430530,0.444913,16.663597,9.834779,18.572694,19.689437
375,2024,3478,Le Moyne,-1.0,-1.0,0.562500,82.7,97.2,-14.5,0.1339,66.5,-9.8,NaN,NaN,-0.988127,-0.134376,0.817137,0.951513,67.530652,0.055646,23.057138,10.392918,0.562500,0.410291,0.450419,20.051728,15.506298,19.611317,15.613674
376,2024,3479,Mercyhurst,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [37]:
df.loc[df['Team Win%'].isna(), :]

,Season,TeamID,Team,Past Year Tournament Result,Past 4 Years Tournament Results,WIN%,ADJOE,ADJDE,ADJEM,BARTHAG,ADJ T.,WAB,Past Year Efficiency Margin,Past 4 Years Efficiency Margin,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo,Starters,Mu,OS Rating,Team Win%,Team EFG%,Opponent EFG%,Team TOR,Team ORBR,Team FTR,Opponent FTR
8,2024,3109,Alliant Intl,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
17,2024,3118,Armstrong St,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20,2024,3121,Augusta,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
27,2024,3128,Birmingham So,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
33,2024,3134,Brooklyn,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
46,2024,3147,Centenary,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
113,2024,3215,Hardin-Simmons,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
114,2024,3216,Hartford,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.505933,-0.326890,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
187,2024,3289,Morris Brown,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
200,2024,3302,NE Illinois,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Map to Matchups

In [38]:
# df_seeds = pd.read_csv(fr'..\data\unprocessed\kaggle\{season}_tourney_seeds.csv')

# df_seeds = df_seeds.loc[df_seeds['Tournament'] == 'M', :].reset_index(drop=True)

# df_seeds.rename(columns={'Seed': 'Region Seed'}, inplace=True)
# df_seeds.insert(2, 'Region', df_seeds['Region Seed'].str[0])
# df_seeds.insert(3, 'Seed', df_seeds['Region Seed'].str.extract('(\d+)').astype(int))

# df_seeds

In [39]:
df_seeds = pd.read_csv(r'..\data\unprocessed\kaggle\WNCAATourneySeeds.csv')

df_seeds = df_seeds.loc[df_seeds['Season'] == season, :].reset_index(drop=True)

df_seeds.insert(2, 'Play In', df_seeds['Seed'].str.endswith(('a', 'b')))
df_seeds.insert(2, 'Region', df_seeds['Seed'].str[0])
df_seeds['Seed'] = df_seeds['Seed'].str.extract('(\d+)').astype(int)

df_seeds = df_seeds.loc[~df_seeds['TeamID'].isin(playin_losers), :].reset_index(drop=True)

df_seeds

,Season,Seed,Region,Play In,TeamID
0,2024,1,W,False,3376
1,2024,2,W,False,3323
2,2024,3,W,False,3333
3,2024,4,W,False,3231
4,2024,5,W,False,3328
...,...,...,...,...,...
59,2024,12,Z,True,3435
60,2024,13,Z,False,3267
61,2024,14,Z,False,3238
62,2024,15,Z,False,3263


Remap to Team A / Team B format

In [40]:
id_to_region = dict(zip(df_seeds['TeamID'], df_seeds['Region']))
id_to_seed = dict(zip(df_seeds['TeamID'], df_seeds['Seed']))

df_mod = pd.DataFrame(
    [
        (team_a, team_b) 
        for team_a in df_seeds['TeamID'].unique() 
        for team_b in df_seeds['TeamID'].unique() 
        if team_a != team_b
    ],
    columns=['Team A ID', 'Team B ID']
)

df_mod.insert(0, 'Season', season)
df_mod['Team A Region'] = df_mod['Team A ID'].map(id_to_region)
df_mod['Team B Region'] = df_mod['Team B ID'].map(id_to_region)
df_mod['Team A Seed'] = df_mod['Team A ID'].map(id_to_seed)
df_mod['Team B Seed'] = df_mod['Team B ID'].map(id_to_seed)

df_mod

,Season,Team A ID,Team B ID,Team A Region,Team B Region,Team A Seed,Team B Seed
0,2024,3376,3323,W,W,1,2
1,2024,3376,3333,W,W,1,3
2,2024,3376,3231,W,W,1,4
3,2024,3376,3328,W,W,1,5
4,2024,3376,3304,W,W,1,6
...,...,...,...,...,...,...,...
4027,2024,3394,3112,Z,Z,16,11
4028,2024,3394,3435,Z,Z,16,12
4029,2024,3394,3267,Z,Z,16,13
4030,2024,3394,3238,Z,Z,16,14


Get round of matchup

In [41]:
same_region = df_mod['Team A Region'] == df_mod['Team B Region']

# round_0_condition = (df_mod['team0_playin'] == 1) & (df_mod['team1_playin'] == 1)  # no play-in games in this data

round_1_condition = df_mod['Team A Seed'] + df_mod['Team B Seed'] == 17

round_2_condition = (
    (df_mod['Team A Seed'].isin([1, 16]) & df_mod['Team B Seed'].isin([8, 9])) | 
    (df_mod['Team A Seed'].isin([8, 9]) & df_mod['Team B Seed'].isin([1, 16])) |
    (df_mod['Team A Seed'].isin([5, 12]) & df_mod['Team B Seed'].isin([4, 13])) | 
    (df_mod['Team A Seed'].isin([4, 13]) & df_mod['Team B Seed'].isin([5, 12])) |
    (df_mod['Team A Seed'].isin([6, 11]) & df_mod['Team B Seed'].isin([3, 14])) | 
    (df_mod['Team A Seed'].isin([3, 14]) & df_mod['Team B Seed'].isin([6, 11])) |
    (df_mod['Team A Seed'].isin([7, 10]) & df_mod['Team B Seed'].isin([2, 15])) | 
    (df_mod['Team A Seed'].isin([2, 15]) & df_mod['Team B Seed'].isin([7, 10]))
)

round_3_condition = (
    (df_mod['Team A Seed'].isin([1, 16, 8, 9]) & df_mod['Team B Seed'].isin([5, 12, 4, 13])) | 
    (df_mod['Team A Seed'].isin([5, 12, 4, 13]) & df_mod['Team B Seed'].isin([1, 16, 8, 9])) |
    (df_mod['Team A Seed'].isin([6, 11, 3, 14]) & df_mod['Team B Seed'].isin([7, 10, 2, 15])) | 
    (df_mod['Team A Seed'].isin([7, 10, 2, 15]) & df_mod['Team B Seed'].isin([6, 11, 3, 14]))
)

round_4_condition = (
    (df_mod['Team A Seed'].isin([1, 16, 8, 9, 5, 12, 4, 13]) & df_mod['Team B Seed'].isin([6, 11, 3, 14, 7, 10, 2, 15])) | 
    (df_mod['Team A Seed'].isin([6, 11, 3, 14, 7, 10, 2, 15]) & df_mod['Team B Seed'].isin([1, 16, 8, 9, 5, 12, 4, 13]))
)

round_5_condition = (
    (df_mod['Team A Region'].isin(['W']) & df_mod['Team B Region'].isin(['X'])) | 
    (df_mod['Team A Region'].isin(['X']) & df_mod['Team B Region'].isin(['W'])) |
    (df_mod['Team A Region'].isin(['Y']) & df_mod['Team B Region'].isin(['Z'])) | 
    (df_mod['Team A Region'].isin(['Z']) & df_mod['Team B Region'].isin(['Y']))
)

round_6_condition = (
    (df_mod['Team A Region'].isin(['W', 'X']) & df_mod['Team B Region'].isin(['Y', 'Z'])) | 
    (df_mod['Team A Region'].isin(['Y', 'Z']) & df_mod['Team B Region'].isin(['W', 'X'])) 
)

round_6_condition

0       False
1       False
2       False
3       False
4       False
        ...  
4027    False
4028    False
4029    False
4030    False
4031    False
Length: 4032, dtype: bool

In [42]:
df_mod['Round'] = -1

df_mod.loc[round_6_condition, 'Round'] = 6

df_mod.loc[round_5_condition, 'Round'] = 5

df_mod.loc[round_4_condition & same_region, 'Round'] = 4

df_mod.loc[round_3_condition & same_region, 'Round'] = 3

df_mod.loc[round_2_condition & same_region, 'Round'] = 2

df_mod.loc[round_1_condition & same_region, 'Round'] = 1

df_mod['Round'].describe()

count    4032.000000
mean        5.095238
std         1.191576
min         1.000000
25%         5.000000
50%         6.000000
75%         6.000000
max         6.000000
Name: Round, dtype: float64

In [43]:
assert ((df_mod['Round'] >= 1).all()), 'The round mapping is incorrect'

Get home court advantage

In [44]:
df_mod['Location'] = 0

# conditions: after 2012 but not 2021 (covid stadium), matchup is within first 2 rounds, and team is top 4 seed
df_mod.loc[
    (df_mod['Season'] > 2012) & 
    (df_mod['Season'] != 2021) & 
    (df_mod['Round'] <= 2) & 
    (df_mod['Team A Seed'] <= 4), 
    'Location'
] = 1

df_mod.loc[
    (df_mod['Season'] > 2012) & 
    (df_mod['Season'] != 2021) & 
    (df_mod['Round'] <= 2) & 
    (df_mod['Team B Seed'] <= 4), 
    'Location'
] = -1

df_mod

,Season,Team A ID,Team B ID,Team A Region,Team B Region,Team A Seed,Team B Seed,Round,Location
0,2024,3376,3323,W,W,1,2,4,0
1,2024,3376,3333,W,W,1,3,4,0
2,2024,3376,3231,W,W,1,4,3,0
3,2024,3376,3328,W,W,1,5,3,0
4,2024,3376,3304,W,W,1,6,4,0
...,...,...,...,...,...,...,...,...,...
4027,2024,3394,3112,Z,Z,16,11,4,0
4028,2024,3394,3435,Z,Z,16,12,3,0
4029,2024,3394,3267,Z,Z,16,13,3,0
4030,2024,3394,3238,Z,Z,16,14,4,0


In [45]:
df_mod['Seed'] = df_mod['Team A Seed'] - df_mod['Team B Seed']

# df_mod.drop(columns=['Team A Region', 'Team B Region', 'Team A Seed', 'Team B Seed'], inplace=True)

df_mod

,Season,Team A ID,Team B ID,Team A Region,Team B Region,Team A Seed,Team B Seed,Round,Location,Seed
0,2024,3376,3323,W,W,1,2,4,0,-1
1,2024,3376,3333,W,W,1,3,4,0,-2
2,2024,3376,3231,W,W,1,4,3,0,-3
3,2024,3376,3328,W,W,1,5,3,0,-4
4,2024,3376,3304,W,W,1,6,4,0,-5
...,...,...,...,...,...,...,...,...,...,...
4027,2024,3394,3112,Z,Z,16,11,4,0,5
4028,2024,3394,3435,Z,Z,16,12,3,0,4
4029,2024,3394,3267,Z,Z,16,13,3,0,3
4030,2024,3394,3238,Z,Z,16,14,4,0,2


Get Head-to-Head

In [46]:
df_h2h = pd.read_parquet('../data/preprocessed/womens_h2h/h2h.parquet')

df_h2h = df_h2h.loc[df_h2h['Season'] == season, :].reset_index(drop=True)

df_h2h

,Season,Team A,Team B,Head to Head,Common Opps
0,2024,Abilene Christian,Alabama,NaN,-1.242485
1,2024,Abilene Christian,Albany (NY),NaN,0.000000
2,2024,Abilene Christian,Alcorn State,NaN,-0.308721
3,2024,Abilene Christian,American,NaN,0.882921
4,2024,Abilene Christian,Arizona,NaN,-0.701317
...,...,...,...,...,...
66407,2024,Youngstown State,Wichita State,NaN,-0.081696
66408,2024,Youngstown State,Wisconsin,NaN,-0.729834
66409,2024,Youngstown State,Wright State,0.023094,-0.172887
66410,2024,Youngstown State,Wyoming,NaN,-0.684239


In [47]:
df_match, team_to_spelling = match_names(df_spellings['TeamNameSpelling'].unique(), df_h2h['Team A'].unique())

df_match.head(25)

  0%|          | 0/360 [00:00<?, ?it/s]

,New Data Team,Team Spelling,Match Score
0,Abilene Christian,abilene christian,100
1,Queens (NC),queens (nc),100
2,Purdue Fort Wayne,purdue fort wayne,100
3,Purdue,purdue,100
4,Providence,providence,100
5,Princeton,princeton,100
6,Presbyterian,presbyterian,100
7,Prairie View,prairie view,100
8,Portland State,portland state,100
9,Quinnipiac,quinnipiac,100


In [48]:
df_h2h.insert(df_h2h.columns.get_loc('Team A'), 'Team A ID', df_h2h['Team A'].map(team_to_spelling).map(spelling_to_id))

df_h2h.insert(df_h2h.columns.get_loc('Team B'), 'Team B ID', df_h2h['Team B'].map(team_to_spelling).map(spelling_to_id))

df_h2h

,Season,Team A ID,Team A,Team B ID,Team B,Head to Head,Common Opps
0,2024,3101,Abilene Christian,3104,Alabama,NaN,-1.242485
1,2024,3101,Abilene Christian,3107,Albany (NY),NaN,0.000000
2,2024,3101,Abilene Christian,3108,Alcorn State,NaN,-0.308721
3,2024,3101,Abilene Christian,3110,American,NaN,0.882921
4,2024,3101,Abilene Christian,3112,Arizona,NaN,-0.701317
...,...,...,...,...,...,...,...
66407,2024,3464,Youngstown State,3455,Wichita State,NaN,-0.081696
66408,2024,3464,Youngstown State,3458,Wisconsin,NaN,-0.729834
66409,2024,3464,Youngstown State,3460,Wright State,0.023094,-0.172887
66410,2024,3464,Youngstown State,3461,Wyoming,NaN,-0.684239


In [49]:
df_mod = pd.merge(
    df_mod,
    df_h2h[['Season', 'Team A ID', 'Team B ID', 'Head to Head', 'Common Opps']],
    how='left',
    on=['Season', 'Team A ID', 'Team B ID'],
)

df_mod

,Season,Team A ID,Team B ID,Team A Region,Team B Region,Team A Seed,Team B Seed,Round,Location,Seed,Head to Head,Common Opps
0,2024,3376,3323,W,W,1,2,4,0,-1,0.8,0.323052
1,2024,3376,3333,W,W,1,3,4,0,-2,NaN,-0.050225
2,2024,3376,3231,W,W,1,4,3,0,-3,NaN,-0.052269
3,2024,3376,3328,W,W,1,5,3,0,-4,NaN,0.986591
4,2024,3376,3304,W,W,1,6,4,0,-5,NaN,0.133184
...,...,...,...,...,...,...,...,...,...,...,...,...
4027,2024,3394,3112,Z,Z,16,11,4,0,5,NaN,NaN
4028,2024,3394,3435,Z,Z,16,12,3,0,4,NaN,-1.371735
4029,2024,3394,3267,Z,Z,16,13,3,0,3,NaN,-0.129032
4030,2024,3394,3238,Z,Z,16,14,4,0,2,NaN,-0.523708


Get team names

In [50]:
df_teams = pd.read_csv(r'..\data\unprocessed\kaggle\WTeams.csv')

df_teams

,TeamID,TeamName
0,3101,Abilene Chr
1,3102,Air Force
2,3103,Akron
3,3104,Alabama
4,3105,Alabama A&M
...,...,...
373,3476,Stonehill
374,3477,East Texas A&M
375,3478,Le Moyne
376,3479,Mercyhurst


In [51]:
id_to_team = dict(zip(df_teams['TeamID'], df_teams['TeamName']))

df_mod.insert(df_mod.columns.get_loc('Team A ID') + 1, 'Team A', df_mod['Team A ID'].map(id_to_team))
df_mod.insert(df_mod.columns.get_loc('Team B ID') + 1, 'Team B', df_mod['Team B ID'].map(id_to_team))

df_mod

,Season,Team A ID,Team A,Team B ID,Team B,Team A Region,Team B Region,Team A Seed,Team B Seed,Round,Location,Seed,Head to Head,Common Opps
0,2024,3376,South Carolina,3323,Notre Dame,W,W,1,2,4,0,-1,0.8,0.323052
1,2024,3376,South Carolina,3333,Oregon St,W,W,1,3,4,0,-2,NaN,-0.050225
2,2024,3376,South Carolina,3231,Indiana,W,W,1,4,3,0,-3,NaN,-0.052269
3,2024,3376,South Carolina,3328,Oklahoma,W,W,1,5,3,0,-4,NaN,0.986591
4,2024,3376,South Carolina,3304,Nebraska,W,W,1,6,4,0,-5,NaN,0.133184
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4027,2024,3394,TAM C. Christi,3112,Arizona,Z,Z,16,11,4,0,5,NaN,NaN
4028,2024,3394,TAM C. Christi,3435,Vanderbilt,Z,Z,16,12,3,0,4,NaN,-1.371735
4029,2024,3394,TAM C. Christi,3267,Marshall,Z,Z,16,13,3,0,3,NaN,-0.129032
4030,2024,3394,TAM C. Christi,3238,Jackson St,Z,Z,16,14,4,0,2,NaN,-0.523708


Map features

In [52]:
team_a_features = pd.merge(
    df_mod[['Season', 'Team A ID']],
    df.drop(columns=['Team']),
    how='left',
    left_on=['Season', 'Team A ID'],
    right_on=['Season', 'TeamID'],
).drop(columns=['Season', 'Team A ID', 'TeamID'])

team_b_features = pd.merge(
    df_mod[['Season', 'Team B ID']],
    df.drop(columns=['Team']),
    how='left',
    left_on=['Season', 'Team B ID'],
    right_on=['Season', 'TeamID'],
).drop(columns=['Season', 'Team B ID', 'TeamID'])

df_features = team_a_features - team_b_features

df_features['Team A ADJOE Team B ADJDE'] = team_a_features['ADJOE'] + team_b_features['ADJDE']
df_features['Team B ADJOE Team A ADJDE'] = team_b_features['ADJOE'] + team_a_features['ADJDE']

df_features['Team A Offense Team B Defense'] = team_a_features['Adjusted Offense'] + team_b_features['Adjusted Defense']
df_features['Team B Offense Team A Defense'] = team_b_features['Adjusted Offense'] + team_a_features['Adjusted Defense']

df_features['Team A Efficiency Margin'] = team_a_features['Efficiency Margin']
df_features['Team B Efficiency Margin'] = team_b_features['Efficiency Margin']

df_features

,Past Year Tournament Result,Past 4 Years Tournament Results,WIN%,ADJOE,ADJDE,ADJEM,BARTHAG,ADJ T.,WAB,Past Year Efficiency Margin,Past 4 Years Efficiency Margin,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo,Starters,Mu,OS Rating,Team Win%,Team EFG%,Opponent EFG%,Team TOR,Team ORBR,Team FTR,Opponent FTR,Team A ADJOE Team B ADJDE,Team B ADJOE Team A ADJDE,Team A Offense Team B Defense,Team B Offense Team A Defense,Team A Efficiency Margin,Team B Efficiency Margin
0,2.0,3.666667,0.187500,12.1,-2.2,14.3,0.0096,-0.5,6.2,0.218675,0.268666,1.813536,0.143240,0.111689,-0.031551,-0.772323,0.395579,8.947108,7.693077,0.187500,0.047505,-0.056051,-1.061853,3.408742,-2.134985,-0.799940,197.9,183.6,1.889501,1.746261,0.540655,0.397414
1,5.0,5.000000,0.225806,10.1,-6.8,16.9,0.0168,5.3,6.9,0.374806,0.259773,1.813554,0.201245,0.098468,-0.102777,5.666187,0.470491,12.978101,12.162396,0.225806,0.014910,-0.036465,-0.586543,5.439675,1.698984,0.722661,202.5,185.6,1.960726,1.759482,0.540655,0.339410
2,3.0,2.666667,0.172414,5.5,-9.2,14.7,0.0147,3.2,7.6,0.164738,0.149010,2.360862,0.187943,0.056169,-0.131774,3.850398,0.561958,13.999782,13.772193,0.172414,-0.036193,-0.087811,-0.531228,7.925315,-0.891497,-3.188423,204.9,190.2,1.989724,1.801781,0.540655,0.352711
3,3.0,4.333333,0.290323,14.0,-8.5,22.5,0.0345,-3.5,8.8,0.312947,0.309540,2.626875,0.259969,0.140664,-0.119304,-3.321241,0.505217,15.282589,14.248288,0.290323,0.060606,-0.078779,-2.942566,1.600584,3.652996,-4.678140,204.2,181.7,1.977254,1.717286,0.540655,0.280686
4,5.0,5.333333,0.333333,13.9,-10.2,24.1,0.0433,3.5,11.2,0.329226,0.282777,3.052043,0.259389,0.132152,-0.127238,3.619005,0.429765,17.878950,16.702941,0.333333,0.053260,-0.097758,-1.848205,1.090692,-1.072356,-2.473125,205.9,181.8,1.985188,1.725798,0.540655,0.281265
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4027,-2.0,-3.333333,0.158249,-17.6,6.9,-24.5,-0.4497,0.8,-6.7,-0.331752,-0.400097,-2.251776,-0.269232,-0.170142,0.099090,0.821007,-0.205369,-5.774637,-6.953271,0.158249,-0.036936,-0.073687,3.567177,3.056323,4.304187,-2.480540,170.2,194.7,1.653559,1.922790,-0.004690,0.264542
4028,0.0,0.000000,-0.015046,-14.1,5.6,-19.7,-0.4163,1.9,-7.1,-0.139232,-0.183700,-1.911751,-0.204524,-0.129500,0.075024,1.740785,-0.152883,-5.672805,-6.318518,-0.015046,-0.021375,-0.047579,2.924659,-1.566177,6.476115,1.438266,171.5,191.2,1.677624,1.882148,-0.004690,0.199835
4029,0.0,0.000000,-0.096296,-15.3,-4.7,-10.6,-0.2730,-6.9,-3.0,-0.025618,-0.076145,-0.816758,-0.160064,-0.169755,-0.009691,-6.671328,-0.133895,-9.370745,-10.082250,-0.096296,-0.062453,-0.083221,2.648865,-2.178235,5.303570,-4.467634,181.8,192.4,1.762339,1.922403,-0.004690,0.155375
4030,0.0,-0.666667,-0.089400,-8.2,-0.4,-7.8,-0.2240,2.4,-4.7,-0.130548,-0.151868,-0.867828,-0.097081,-0.078461,0.018620,1.494852,-0.115386,-3.571311,-2.804590,-0.089400,0.006565,0.024867,2.288782,-3.901513,-4.138111,-4.680749,177.5,185.3,1.734028,1.831110,-0.004690,0.092392


In [53]:
df_mod[df_features.columns] = df_features

df_mod

,Season,Team A ID,Team A,Team B ID,Team B,Team A Region,Team B Region,Team A Seed,Team B Seed,Round,Location,Seed,Head to Head,Common Opps,Past Year Tournament Result,Past 4 Years Tournament Results,WIN%,ADJOE,ADJDE,ADJEM,BARTHAG,ADJ T.,WAB,Past Year Efficiency Margin,Past 4 Years Efficiency Margin,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo,Starters,Mu,OS Rating,Team Win%,Team EFG%,Opponent EFG%,Team TOR,Team ORBR,Team FTR,Opponent FTR,Team A ADJOE Team B ADJDE,Team B ADJOE Team A ADJDE,Team A Offense Team B Defense,Team B Offense Team A Defense,Team A Efficiency Margin,Team B Efficiency Margin
0,2024,3376,South Carolina,3323,Notre Dame,W,W,1,2,4,0,-1,0.8,0.323052,2.0,3.666667,0.187500,12.1,-2.2,14.3,0.0096,-0.5,6.2,0.218675,0.268666,1.813536,0.143240,0.111689,-0.031551,-0.772323,0.395579,8.947108,7.693077,0.187500,0.047505,-0.056051,-1.061853,3.408742,-2.134985,-0.799940,197.9,183.6,1.889501,1.746261,0.540655,0.397414
1,2024,3376,South Carolina,3333,Oregon St,W,W,1,3,4,0,-2,NaN,-0.050225,5.0,5.000000,0.225806,10.1,-6.8,16.9,0.0168,5.3,6.9,0.374806,0.259773,1.813554,0.201245,0.098468,-0.102777,5.666187,0.470491,12.978101,12.162396,0.225806,0.014910,-0.036465,-0.586543,5.439675,1.698984,0.722661,202.5,185.6,1.960726,1.759482,0.540655,0.339410
2,2024,3376,South Carolina,3231,Indiana,W,W,1,4,3,0,-3,NaN,-0.052269,3.0,2.666667,0.172414,5.5,-9.2,14.7,0.0147,3.2,7.6,0.164738,0.149010,2.360862,0.187943,0.056169,-0.131774,3.850398,0.561958,13.999782,13.772193,0.172414,-0.036193,-0.087811,-0.531228,7.925315,-0.891497,-3.188423,204.9,190.2,1.989724,1.801781,0.540655,0.352711
3,2024,3376,South Carolina,3328,Oklahoma,W,W,1,5,3,0,-4,NaN,0.986591,3.0,4.333333,0.290323,14.0,-8.5,22.5,0.0345,-3.5,8.8,0.312947,0.309540,2.626875,0.259969,0.140664,-0.119304,-3.321241,0.505217,15.282589,14.248288,0.290323,0.060606,-0.078779,-2.942566,1.600584,3.652996,-4.678140,204.2,181.7,1.977254,1.717286,0.540655,0.280686
4,2024,3376,South Carolina,3304,Nebraska,W,W,1,6,4,0,-5,NaN,0.133184,5.0,5.333333,0.333333,13.9,-10.2,24.1,0.0433,3.5,11.2,0.329226,0.282777,3.052043,0.259389,0.132152,-0.127238,3.619005,0.429765,17.878950,16.702941,0.333333,0.053260,-0.097758,-1.848205,1.090692,-1.072356,-2.473125,205.9,181.8,1.985188,1.725798,0.540655,0.281265
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4027,2024,3394,TAM C. Christi,3112,Arizona,Z,Z,16,11,4,0,5,NaN,NaN,-2.0,-3.333333,0.158249,-17.6,6.9,-24.5,-0.4497,0.8,-6.7,-0.331752,-0.400097,-2.251776,-0.269232,-0.170142,0.099090,0.821007,-0.205369,-5.774637,-6.953271,0.158249,-0.036936,-0.073687,3.567177,3.056323,4.304187,-2.480540,170.2,194.7,1.653559,1.922790,-0.004690,0.264542
4028,2024,3394,TAM C. Christi,3435,Vanderbilt,Z,Z,16,12,3,0,4,NaN,-1.371735,0.0,0.000000,-0.015046,-14.1,5.6,-19.7,-0.4163,1.9,-7.1,-0.139232,-0.183700,-1.911751,-0.204524,-0.129500,0.075024,1.740785,-0.152883,-5.672805,-6.318518,-0.015046,-0.021375,-0.047579,2.924659,-1.566177,6.476115,1.438266,171.5,191.2,1.677624,1.882148,-0.004690,0.199835
4029,2024,3394,TAM C. Christi,3267,Marshall,Z,Z,16,13,3,0,3,NaN,-0.129032,0.0,0.000000,-0.096296,-15.3,-4.7,-10.6,-0.2730,-6.9,-3.0,-0.025618,-0.076145,-0.816758,-0.160064,-0.169755,-0.009691,-6.671328,-0.133895,-9.370745,-10.082250,-0.096296,-0.062453,-0.083221,2.648865,-2.178235,5.303570,-4.467634,181.8,192.4,1.762339,1.922403,-0.004690,0.155375
4030,2024,3394,TAM C. Christi,3238,Jackson St,Z,Z,16,14,4,0,2,NaN,-0.523708,0.0,-0.666667,-0.089400,-8.2,-0.4,-7.8,-0.2240,2.4,-4.7,-0.130548,-0.151868,-0.867828,-0.097081,-0.078461,0.018620,1.494852,-0.115386,-3.571311,-2.804590,-0.089400,0.006565,0.024867,2.288782,-3.901513,-4.138111,-4.680749,177.5,185.3,1.734028,1.831110,-0.004690,0.092392


In [54]:
df_mod.drop(columns=['Team A Region', 'Team B Region', 'Team A Seed', 'Team B Seed'], inplace=True)

df_mod

,Season,Team A ID,Team A,Team B ID,Team B,Round,Location,Seed,Head to Head,Common Opps,Past Year Tournament Result,Past 4 Years Tournament Results,WIN%,ADJOE,ADJDE,ADJEM,BARTHAG,ADJ T.,WAB,Past Year Efficiency Margin,Past 4 Years Efficiency Margin,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo,Starters,Mu,OS Rating,Team Win%,Team EFG%,Opponent EFG%,Team TOR,Team ORBR,Team FTR,Opponent FTR,Team A ADJOE Team B ADJDE,Team B ADJOE Team A ADJDE,Team A Offense Team B Defense,Team B Offense Team A Defense,Team A Efficiency Margin,Team B Efficiency Margin
0,2024,3376,South Carolina,3323,Notre Dame,4,0,-1,0.8,0.323052,2.0,3.666667,0.187500,12.1,-2.2,14.3,0.0096,-0.5,6.2,0.218675,0.268666,1.813536,0.143240,0.111689,-0.031551,-0.772323,0.395579,8.947108,7.693077,0.187500,0.047505,-0.056051,-1.061853,3.408742,-2.134985,-0.799940,197.9,183.6,1.889501,1.746261,0.540655,0.397414
1,2024,3376,South Carolina,3333,Oregon St,4,0,-2,NaN,-0.050225,5.0,5.000000,0.225806,10.1,-6.8,16.9,0.0168,5.3,6.9,0.374806,0.259773,1.813554,0.201245,0.098468,-0.102777,5.666187,0.470491,12.978101,12.162396,0.225806,0.014910,-0.036465,-0.586543,5.439675,1.698984,0.722661,202.5,185.6,1.960726,1.759482,0.540655,0.339410
2,2024,3376,South Carolina,3231,Indiana,3,0,-3,NaN,-0.052269,3.0,2.666667,0.172414,5.5,-9.2,14.7,0.0147,3.2,7.6,0.164738,0.149010,2.360862,0.187943,0.056169,-0.131774,3.850398,0.561958,13.999782,13.772193,0.172414,-0.036193,-0.087811,-0.531228,7.925315,-0.891497,-3.188423,204.9,190.2,1.989724,1.801781,0.540655,0.352711
3,2024,3376,South Carolina,3328,Oklahoma,3,0,-4,NaN,0.986591,3.0,4.333333,0.290323,14.0,-8.5,22.5,0.0345,-3.5,8.8,0.312947,0.309540,2.626875,0.259969,0.140664,-0.119304,-3.321241,0.505217,15.282589,14.248288,0.290323,0.060606,-0.078779,-2.942566,1.600584,3.652996,-4.678140,204.2,181.7,1.977254,1.717286,0.540655,0.280686
4,2024,3376,South Carolina,3304,Nebraska,4,0,-5,NaN,0.133184,5.0,5.333333,0.333333,13.9,-10.2,24.1,0.0433,3.5,11.2,0.329226,0.282777,3.052043,0.259389,0.132152,-0.127238,3.619005,0.429765,17.878950,16.702941,0.333333,0.053260,-0.097758,-1.848205,1.090692,-1.072356,-2.473125,205.9,181.8,1.985188,1.725798,0.540655,0.281265
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4027,2024,3394,TAM C. Christi,3112,Arizona,4,0,5,NaN,NaN,-2.0,-3.333333,0.158249,-17.6,6.9,-24.5,-0.4497,0.8,-6.7,-0.331752,-0.400097,-2.251776,-0.269232,-0.170142,0.099090,0.821007,-0.205369,-5.774637,-6.953271,0.158249,-0.036936,-0.073687,3.567177,3.056323,4.304187,-2.480540,170.2,194.7,1.653559,1.922790,-0.004690,0.264542
4028,2024,3394,TAM C. Christi,3435,Vanderbilt,3,0,4,NaN,-1.371735,0.0,0.000000,-0.015046,-14.1,5.6,-19.7,-0.4163,1.9,-7.1,-0.139232,-0.183700,-1.911751,-0.204524,-0.129500,0.075024,1.740785,-0.152883,-5.672805,-6.318518,-0.015046,-0.021375,-0.047579,2.924659,-1.566177,6.476115,1.438266,171.5,191.2,1.677624,1.882148,-0.004690,0.199835
4029,2024,3394,TAM C. Christi,3267,Marshall,3,0,3,NaN,-0.129032,0.0,0.000000,-0.096296,-15.3,-4.7,-10.6,-0.2730,-6.9,-3.0,-0.025618,-0.076145,-0.816758,-0.160064,-0.169755,-0.009691,-6.671328,-0.133895,-9.370745,-10.082250,-0.096296,-0.062453,-0.083221,2.648865,-2.178235,5.303570,-4.467634,181.8,192.4,1.762339,1.922403,-0.004690,0.155375
4030,2024,3394,TAM C. Christi,3238,Jackson St,4,0,2,NaN,-0.523708,0.0,-0.666667,-0.089400,-8.2,-0.4,-7.8,-0.2240,2.4,-4.7,-0.130548,-0.151868,-0.867828,-0.097081,-0.078461,0.018620,1.494852,-0.115386,-3.571311,-2.804590,-0.089400,0.006565,0.024867,2.288782,-3.901513,-4.138111,-4.680749,177.5,185.3,1.734028,1.831110,-0.004690,0.092392


Check that data follows same format as the data that the model was trained on

In [55]:
df_mod_training = pd.read_parquet(data_path)

assert all(df_mod_training.drop(columns=['Result']).columns == df_mod.columns), 'Columns do not match'

'Columns Match'

'Columns Match'

### Get Model Predictions

In [56]:
import pickle

with open(model_path, 'rb') as f:
    mod = pickle.load(f)

mod

LGBMClassifier(early_stopping_round=25, feature_fraction=0.6927277599076377,
               lambda_l1=9.058505139074182, lambda_l2=5.2722264970292905,
               max_depth=21, metric='rmse', min_child_samples=1,
               monotone_constraints=[0, 1, -1, 1, 1, 1, 1, 1, 1, -1, 1, 1, 1, 1,
                                     1, 1, 1, 1, 1, -1, 1, 1, 1, 1, 1, 1, -1,
                                     -1, 1, 0, ...],
               n_estimators=500, num_leaves=87, random_state=22, verbosity=-1)

In [57]:
X = df_mod.drop(columns=['Season', 'Team A ID', 'Team A', 'Team B ID', 'Team B'])

predictions = mod.predict_proba(X)[:, 1]

predictions

array([0.9375549 , 0.92383085, 0.9137376 , ..., 0.1960915 , 0.18679329,
       0.21623746])

Turn predictions into matchup matrix

In [58]:
df_matrix = (
    df_mod[['Team A ID', 'Team B ID']]
    .assign(Prediction=predictions)
    .pivot(
        index=['Team A ID'], 
        columns=['Team B ID'],
        values='Prediction',
    )
)

df_matrix

Team B ID,3104,3112,3124,3151,3160,3163,3166,3179,3180,3181,3186,3193,3195,3199,3211,3231,3234,3235,3238,3242,3243,3245,3257,3261,3263,3266,3267,3268,3276,3277,3279,3292,3301,3304,3313,3314,3323,3326,3328,3333,3339,3342,3343,3349,3350,3355,3376,3390,3393,3394,3397,3400,3401,3404,3414,3417,3424,3425,3428,3435,3439,3452,3453,3465
Team A ID,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
3104,NaN,0.532997,0.249733,0.911312,0.287490,0.060767,0.263217,0.757277,0.939226,0.339851,0.843883,0.769793,0.755994,0.713223,0.159887,0.199885,0.129346,0.323671,0.901078,0.556624,0.328798,0.912412,0.427827,0.126153,0.906912,0.720253,0.875179,0.305022,0.592935,0.298313,0.258396,0.650989,0.250148,0.363112,0.913919,0.701439,0.189810,0.227775,0.417078,0.247650,0.904595,0.945322,0.271104,0.910248,0.687948,0.725120,0.044177,0.086894,0.411898,0.935744,0.192831,0.096851,0.787665,0.945322,0.901803,0.095206,0.400177,0.114796,0.251800,0.806672,0.237100,0.286561,0.632046,0.909813
3112,0.480005,NaN,0.189306,0.898269,0.209489,0.060374,0.340888,0.732107,0.938253,0.424383,0.886452,0.783818,0.587739,0.534685,0.148066,0.232907,0.171487,0.294280,0.897776,0.564971,0.390246,0.905568,0.219911,0.188280,0.904404,0.676189,0.838644,0.309033,0.701832,0.390467,0.326349,0.614654,0.178955,0.381232,0.900449,0.642775,0.225943,0.168489,0.356969,0.177679,0.911950,0.939226,0.290089,0.905390,0.696587,0.693748,0.053877,0.097138,0.622267,0.936615,0.273165,0.056927,0.666493,0.945322,0.904336,0.059718,0.356105,0.173255,0.478563,0.772999,0.220346,0.323573,0.679878,0.887245
3124,0.748752,0.803946,NaN,0.937486,0.548478,0.208890,0.607361,0.833349,0.947569,0.685611,0.937773,0.852589,0.829175,0.823742,0.439406,0.297466,0.330532,0.449882,0.941533,0.609436,0.427169,0.947569,0.575627,0.380014,0.941533,0.839137,0.906775,0.692016,0.819544,0.643562,0.778328,0.795038,0.325023,0.743341,0.941708,0.803509,0.440911,0.373830,0.571296,0.532993,0.945322,0.947569,0.644335,0.947569,0.754775,0.834375,0.104429,0.208680,0.756305,0.947569,0.645220,0.211807,0.830868,0.947569,0.938335,0.287360,0.688819,0.429310,0.652405,0.820050,0.471147,0.760014,0.863159,0.937486
3151,0.082206,0.094270,0.056459,NaN,0.061657,0.053877,0.071359,0.190050,0.529344,0.065247,0.344508,0.151159,0.165485,0.094919,0.064937,0.060211,0.054683,0.100120,0.315344,0.100694,0.056251,0.590250,0.066844,0.053877,0.579655,0.103103,0.164650,0.093328,0.174084,0.079225,0.082647,0.144620,0.056459,0.111986,0.694115,0.094406,0.053877,0.056459,0.070057,0.056459,0.378566,0.919002,0.076514,0.346016,0.080298,0.173390,0.053877,0.056459,0.097231,0.770115,0.082769,0.053877,0.179794,0.828771,0.558375,0.056251,0.160360,0.053877,0.061657,0.190954,0.063989,0.062691,0.174638,0.413474
3160,0.728141,0.800520,0.431396,0.935743,NaN,0.179666,0.548692,0.856991,0.941708,0.654539,0.933405,0.836665,0.860307,0.772893,0.281971,0.319561,0.214886,0.705190,0.933405,0.764086,0.694905,0.941405,0.464705,0.437306,0.933566,0.777216,0.880960,0.552544,0.765931,0.629560,0.718388,0.845459,0.275151,0.703314,0.941405,0.763888,0.365402,0.285737,0.713073,0.481539,0.934423,0.945976,0.626299,0.932137,0.794507,0.859738,0.079313,0.361716,0.768334,0.941533,0.575595,0.178928,0.830583,0.947569,0.930024,0.136005,0.806238,0.198768,0.487155,0.836120,0.393977,0.659902,0.876468,0.935899
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3435,0.155184,0.220001,0.162151,0.792756,0.151203,0.057468,0.235429,0.446588,0.903269,0.219190,0.750618,0.788828,0.272283,0.224193,0.201882,0.197371,0.089477,0.372586,0.761656,0.263710,0.154109,0.885941,0.200270,0.096008,0.877987,0.284474,0.619029,0.225570,0.264431,0.235325,0.121804,0.273877,0.104532,0.230050,0.887861,0.235213,0.066831,0.135847,0.249267,0.075928,0.829524,0.937329,0.196449,0.884709,0.321448,0.489528,0.046742,0.058325,0.232032,0.904378,0

In [59]:
df_matrix_display = df_matrix.copy()

df_matrix_display.columns = df_matrix_display.columns.map(id_to_team)
df_matrix_display.index = df_matrix_display.index.map(id_to_team)

df_matrix_display

Team B ID,Alabama,Arizona,Baylor,Chattanooga,Colorado,Connecticut,Creighton,Drake,Drexel,Duke,E Washington,Fairfield,FGCU,Florida St,Gonzaga,Indiana,Iowa,Iowa St,Jackson St,Kansas,Kansas St,Kent,Louisville,LSU,Maine,Marquette,Marshall,Maryland,Michigan,Michigan St,Mississippi,MTSU,NC State,Nebraska,Norfolk St,North Carolina,Notre Dame,Ohio St,Oklahoma,Oregon St,Portland,Presbyterian,Princeton,Rice,Richmond,S Dakota St,South Carolina,Stanford,Syracuse,TAM C. Christi,Tennessee,Texas,Texas A&M,TN Martin,UC Irvine,UCLA,UNLV,USC,Utah,Vanderbilt,Virginia Tech,West Virginia,WI Green Bay,Cal Baptist
Team A ID,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
Alabama,NaN,0.532997,0.249733,0.911312,0.287490,0.060767,0.263217,0.757277,0.939226,0.339851,0.843883,0.769793,0.755994,0.713223,0.159887,0.199885,0.129346,0.323671,0.901078,0.556624,0.328798,0.912412,0.427827,0.126153,0.906912,0.720253,0.875179,0.305022,0.592935,0.298313,0.258396,0.650989,0.250148,0.363112,0.913919,0.701439,0.189810,0.227775,0.417078,0.247650,0.904595,0.945322,0.271104,0.910248,0.687948,0.725120,0.044177,0.086894,0.411898,0.935744,0.192831,0.096851,0.787665,0.945322,0.901803,0.095206,0.400177,0.114796,0.251800,0.806672,0.237100,0.286561,0.632046,0.909813
Arizona,0.480005,NaN,0.189306,0.898269,0.209489,0.060374,0.340888,0.732107,0.938253,0.424383,0.886452,0.783818,0.587739,0.534685,0.148066,0.232907,0.171487,0.294280,0.897776,0.564971,0.390246,0.905568,0.219911,0.188280,0.904404,0.676189,0.838644,0.309033,0.701832,0.390467,0.326349,0.614654,0.178955,0.381232,0.900449,0.642775,0.225943,0.168489,0.356969,0.177679,0.911950,0.939226,0.290089,0.905390,0.696587,0.693748,0.053877,0.097138,0.622267,0.936615,0.273165,0.056927,0.666493,0.945322,0.904336,0.059718,0.356105,0.173255,0.478563,0.772999,0.220346,0.323573,0.679878,0.887245
Baylor,0.748752,0.803946,NaN,0.937486,0.548478,0.208890,0.607361,0.833349,0.947569,0.685611,0.937773,0.852589,0.829175,0.823742,0.439406,0.297466,0.330532,0.449882,0.941533,0.609436,0.427169,0.947569,0.575627,0.380014,0.941533,0.839137,0.906775,0.692016,0.819544,0.643562,0.778328,0.795038,0.325023,0.743341,0.941708,0.803509,0.440911,0.373830,0.571296,0.532993,0.945322,0.947569,0.644335,0.947569,0.754775,0.834375,0.104429,0.208680,0.756305,0.947569,0.645220,0.211807,0.830868,0.947569,0.938335,0.287360,0.688819,0.429310,0.652405,0.820050,0.471147,0.760014,0.863159,0.937486
Chattanooga,0.082206,0.094270,0.056459,NaN,0.061657,0.053877,0.071359,0.190050,0.529344,0.065247,0.344508,0.151159,0.165485,0.094919,0.064937,0.060211,0.054683,0.100120,0.315344,0.100694,0.056251,0.590250,0.066844,0.053877,0.579655,0.103103,0.164650,0.093328,0.174084,0.079225,0.082647,0.144620,0.056459,0.111986,0.694115,0.094406,0.053877,0.056459,0.070057,0.056459,0.378566,0.919002,0.076514,0.346016,0.080298,0.173390,0.053877,0.056459,0.097231,0.770115,0.082769,0.053877,0.179794,0.828771,0.558375,0.056251,0.160360,0.053877,0.061657,0.190954,0.063989,0.062691,0.174638,0.413474
Colorado,0.728141,0.800520,0.431396,0.935743,NaN,0.179666,0.548692,0.856991,0.941708,0.654539,0.933405,0.836665,0.860307,0.772893,0.281971,0.319561,0.214886,0.705190,0.933405,0.764086,0.694905,0.941405,0.464705,0.437306,0.933566,0.777216,0.880960,0.552544,0.765931,0.629560,0.718388,0.845459,0.275151,0.703314,0.941405,0.763888,0.365402,0.285737,0.713073,0.481539,0.934423,0.945976,0.626299,0.932137,0.794507,0.859738,0.079313,0.361716,0.768334,0.941533,0.575595,0.178928,0.830583,0.947569,0.930024,0.136005,0.806238,0.198768,0.487155,0.836120,0.393977,0.659902,0.876468,0.935899
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Vanderbilt,0.155184,0.220001,0.162151,0.792756,0.151203,0.057468,0.235429,0.446588,0.903269,0.219190,0.750618,0.788828,0.272283,0.224193,0.201882,0.197371,0.089477,0.37

In [61]:
df_matrix.to_csv(f'../data/simulations/womens/matchup_matrix_{season}.csv', index=True)
df_matrix_display.to_csv(f'../data/simulations/womens/matchup_matrix_display_{season}.csv', index=True)

'Done'

'Done'